# Team Lannisters: Span-based BioNER on Kaggle (training)
Settings (right panel): **Accelerator = GPU T4 x2** (we use one GPU), **Internet = On**.
Add your `bioner-code` dataset via **Add Input**.

**Two ways to run**
- *Interactive*: run the setup cell, then only the cells for the phase you are testing.
- *Final run*: **Save Version -> Save & Run All (Commit)**. It runs every cell in the background (about 30-40 min on a T4), so you can close the browser. All outputs land in the version's **Output** tab.

## Setup: copy code + data into /kaggle/working/bioner

In [ ]:
import glob, os, shutil
W = '/kaggle/working/bioner'
# the bundle is the only input with requirements.txt next to src/ (a training output has a bioner/src copy without it)
cfg = [c for c in glob.glob('/kaggle/input/**/src/config.py', recursive=True)
       if os.path.exists(os.path.join(os.path.dirname(os.path.dirname(c)), 'requirements.txt'))]
assert cfg, 'bioner-code dataset not found: Add Input -> your dataset'
root = os.path.dirname(os.path.dirname(cfg[0]))
for sub in ['src', 'data', 'backend']:                 # refresh code, keep existing checkpoints
    shutil.rmtree(f'{W}/{sub}', ignore_errors=True)
    shutil.copytree(f'{root}/{sub}', f'{W}/{sub}')
ck = glob.glob('/kaggle/input/**/bioner/checkpoints/span/span_model.pt', recursive=True)
if ck and not os.path.exists(f'{W}/checkpoints/span'):   # reuse a finished training run
    prev = ck[0].split('/checkpoints/')[0]
    for sub in ['checkpoints', 'submission']:
        if os.path.exists(f'{prev}/{sub}'):
            shutil.copytree(f'{prev}/{sub}', f'{W}/{sub}', dirs_exist_ok=True)
    print('copied checkpoints from', prev)
%cd /kaggle/working/bioner
!pip install -q seqeval fastapi uvicorn   # torch, transformers, pandas, sklearn are preinstalled
assert _exit_code == 0, 'command failed, see output above'
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'no GPU: running on CPU'
assert _exit_code == 0, 'command failed, see output above'

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4 x2, then restart'

## Phase 1: data tests

In [ ]:
!python src/test_data.py
assert _exit_code == 0, 'command failed, see output above'

## Phase 2: baseline (debug run, then full run)

In [ ]:
!python src/train.py --model baseline --debug
assert _exit_code == 0, 'command failed, see output above'

In [ ]:
!python src/train.py --model baseline
assert _exit_code == 0, 'command failed, see output above'

## Phase 3: span model shape check

In [ ]:
!python src/span_model.py
assert _exit_code == 0, 'command failed, see output above'

## Phase 4: span model (debug run, then full run)

In [ ]:
!python src/sampling.py && python src/decode.py      # tiny self-checks
assert _exit_code == 0, 'command failed, see output above'

In [ ]:
!python src/train.py --model span --debug
assert _exit_code == 0, 'command failed, see output above'

In [ ]:
!python src/train.py --model span
assert _exit_code == 0, 'command failed, see output above'

## Phase 5: tune threshold + compare both models on validation

In [ ]:
!python src/evaluate.py
assert _exit_code == 0, 'command failed, see output above'

## Phase 6: test-set predictions -> submission CSVs

In [ ]:
!python src/predict.py
assert _exit_code == 0, 'command failed, see output above'
!python src/predict.py --model baseline
assert _exit_code == 0, 'command failed, see output above'
!cp submission/submission.csv /kaggle/working/submission.csv   # easy to find in Output
assert _exit_code == 0, 'command failed, see output above'

## Package results for download
Zips `checkpoints/baseline`, `checkpoints/span`, `results.md` and `submission/` into `/kaggle/working/checkpoints.zip` (about 800 MB). Download it from the Output panel, then on your laptop:
```
cd ~/Desktop/PDNC && unzip -o ~/Downloads/checkpoints.zip
```

In [ ]:
!rm -rf checkpoints/debug && cd /kaggle/working && zip -rq checkpoints.zip bioner/checkpoints bioner/results.md bioner/submission && ls -lh checkpoints.zip
assert _exit_code == 0, 'command failed, see output above'